# Imports and Paths

In [1]:
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
import hdf5plugin  # to read the compressed data
import os
import sys
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
import stable_worldmodel as swm  # to load and run LeWM
from stable_worldmodel.data.formats.hdf5 import HDF5Dataset

## Paths and Helper Imports

In [2]:
REPO_ROOT = Path.cwd().parents[1]
LEWM_DIR = REPO_ROOT / "third_party" / "le-wm"
if str(LEWM_DIR) not in sys.path:
    sys.path.insert(0, str(LEWM_DIR))

from utils import get_img_preprocessor

REPO_ROOT = Path.cwd().parents[1] #Have to go up one
H5_PATH = REPO_ROOT / "data" / "processed" / "pusht_expert_train.h5" #Training data provided by SWM
os.environ.setdefault("STABLEWM_HOME", str(REPO_ROOT / "data" / "stablewm"))

device = "cuda" if torch.cuda.is_available() else "cpu"
model = swm.policy.AutoCostModel("pusht/lewm").to(device).eval()

/workspace/Safety-Dial/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


07:49:35 | INFO  | __init__.py | JAX version 0.10.2 available.
07:49:38 | INFO  | atomic_chec~| [atomic_save] installed crash-safe checkpoint plugin (write to sibling .tmp + fsync + atomic rename)


## Helper Funcs

In [3]:
print(device)

cuda


In [4]:
if not H5_PATH.exists():
    raise FileNotFoundError(f"Nothing there, run download_data.py")

In [5]:
# We use the same normalisation that they use in their pipeline during training
prep = get_img_preprocessor(source="pixels", target="pixels", img_size=224)

def pixels_to_tensor(frames_hwc):
    model_device = next(model.parameters()).device
    xs = []
    for f in frames_hwc:
        sample = {"pixels": f}
        prep(sample)
        xs.append(sample["pixels"])
    return torch.stack(xs, 0).unsqueeze(0).to(model_device)

## Exploring the h5

In [6]:
file = h5py.File(H5_PATH, "r")

In [7]:
print(list(file.keys()))

['action', 'ep_len', 'ep_offset', 'episode_idx', 'pixels', 'proprio', 'state', 'step_idx']


In [8]:
for key in file.keys():
    item = file[key]
    print(f"{key:12s}  shape={str(item.shape):25s}  dtype={item.dtype}")

action        shape=(2336736, 2)               dtype=float32
ep_len        shape=(18685,)                   dtype=int32
ep_offset     shape=(18685,)                   dtype=int64
episode_idx   shape=(2336736,)                 dtype=int64
pixels        shape=(2336736, 224, 224, 3)     dtype=uint8
proprio       shape=(2336736, 4)               dtype=float32
state         shape=(2336736, 7)               dtype=float32
step_idx      shape=(2336736,)                 dtype=int64


## Testing out a single episode

In [9]:
# first 64 steps of episode 0
ep = 0
off, length = int(file["ep_offset"][ep]), int(file["ep_len"][ep])
T = min(64, length)  # just in case the episode is shorter
idx = slice(off, off + T)

batch_frames = file["pixels"][idx]  # (T, 224, 224, 3)
y_block = file["state"][idx, 2:4]  # confirm cols vs notes

with torch.no_grad():
    pixels = pixels_to_tensor(batch_frames)
    print(f"model: {next(model.parameters()).device}, pixels: {pixels.device}")
    info = model.encode({"pixels": pixels})
    # ViT encode runs on GPU when model/pixels are on cuda; .cpu().numpy() is only for numpy/sklearn
    z = info["emb"][0].cpu().numpy()  # (T, 192)

model: cuda:0, pixels: cuda:0


In [10]:
z.shape

(64, 192)

## Linear probe: encoder `z` → block `(x, y)`

Use `stable_worldmodel`'s `HDF5Dataset` + `DataLoader` (same clip indexing as LeWM training).

- **Labels:** `state[..., 2:4]` (block xy — heuristic from notes; validate if r is poor)
- **Split:** by **episode**, not random steps
- **Subsample:** every `PROBE_FRAMESKIP` env step, capped episodes, so encoding finishes in minutes
- **Model:** ridge regression on frozen post-projection latents (`encode` → `emb`)

In [11]:
# Probe data config (match training frameskip=5 for observation rate; keep num_steps=1 for single-frame encode)
PROBE_FRAMESKIP = 5
NUM_STEPS = 1
BLOCK_XY_SLICE = slice(2, 4)

N_TRAIN_EPS = 200
N_VAL_EPS = 50
MAX_CLIPS_PER_EP = 16  # after frameskip indexing; keeps encode cheap
BATCH_SIZE = 32
SEED = 0
RIDGE_ALPHA = 1.0

rng = np.random.default_rng(SEED)

In [12]:
# HDF5Dataset opens with swmr=True — close the exploration handle from above first
if "file" in globals() and file.id.valid:
    file.close()

# Same ImageNet preprocess as le-wm training, via their helper (already defined as `prep`)
probe_ds = HDF5Dataset(
    path=str(H5_PATH),
    frameskip=PROBE_FRAMESKIP,
    num_steps=NUM_STEPS,
    keys_to_load=["pixels", "state"],
    transform=prep,
)

n_eps = len(probe_ds.lengths)
print(f"dataset clips={len(probe_ds):,}  episodes={n_eps:,}  frameskip={PROBE_FRAMESKIP}")

# Episode-held-out split
ep_ids = np.arange(n_eps)
rng.shuffle(ep_ids)
train_eps = set(ep_ids[:N_TRAIN_EPS].tolist())
val_eps = set(ep_ids[N_TRAIN_EPS : N_TRAIN_EPS + N_VAL_EPS].tolist())

# clip_indices entries are (ep_idx, start). Keep every frameskip-aligned start once per
# sample window; then cap per episode so we don't encode millions of frames.
train_idx, val_idx = [], []
per_ep_count: dict[int, int] = {}
for i, (ep, start) in enumerate(probe_ds.clip_indices):
    # With frameskip=F, starts 0..F-1 all map to nearly-overlapping windows; take start % F == 0
    if start % PROBE_FRAMESKIP != 0:
        continue
    if per_ep_count.get(ep, 0) >= MAX_CLIPS_PER_EP:
        continue
    if ep in train_eps:
        train_idx.append(i)
        per_ep_count[ep] = per_ep_count.get(ep, 0) + 1
    elif ep in val_eps:
        val_idx.append(i)
        per_ep_count[ep] = per_ep_count.get(ep, 0) + 1

print(f"train clips={len(train_idx):,}  val clips={len(val_idx):,}")
print(f"train eps used={len(train_eps)}  val eps used={len(val_eps)}")

train_loader = DataLoader(
    Subset(probe_ds, train_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
val_loader = DataLoader(
    Subset(probe_ds, val_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

dataset clips=2,261,996  episodes=18,685  frameskip=5
train clips=3,102  val clips=792
train eps used=200  val eps used=50


In [13]:
@torch.no_grad()
def encode_loader(loader: DataLoader):
    """Encode DataLoader batches -> (Z, Y_block) numpy arrays."""
    zs, ys = [], []
    model_device = next(model.parameters()).device
    for batch in loader:
        # batch['pixels']: (B, T, C, H, W) already ImageNet-normalized by `prep`
        pixels = batch["pixels"].to(model_device, non_blocking=True)
        state = batch["state"]  # (B, T, 7) on CPU
        info = model.encode({"pixels": pixels})
        z = info["emb"].reshape(-1, info["emb"].shape[-1]).cpu().numpy()
        y = state[..., BLOCK_XY_SLICE].reshape(-1, 2).numpy()
        zs.append(z)
        ys.append(y)
    return np.concatenate(zs, axis=0), np.concatenate(ys, axis=0)


print("encoding train…")
Z_train, Y_train = encode_loader(train_loader)
print("encoding val…")
Z_val, Y_val = encode_loader(val_loader)
print(f"Z_train={Z_train.shape}  Y_train={Y_train.shape}")
print(f"Z_val={Z_val.shape}  Y_val={Y_val.shape}")

encoding train…


KeyboardInterrupt: 

In [ ]:
probe = Ridge(alpha=RIDGE_ALPHA)
probe.fit(Z_train, Y_train)

Y_hat_train = probe.predict(Z_train)
Y_hat_val = probe.predict(Z_val)


def report(split: str, y_true: np.ndarray, y_pred: np.ndarray):
    r2 = r2_score(y_true, y_pred, multioutput="raw_values")
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=0))
    rx, _ = pearsonr(y_true[:, 0], y_pred[:, 0])
    ry, _ = pearsonr(y_true[:, 1], y_pred[:, 1])
    print(
        f"{split:5s}  r(x)={rx:.4f}  r(y)={ry:.4f}  "
        f"R²(x)={r2[0]:.4f}  R²(y)={r2[1]:.4f}  "
        f"RMSE(x)={rmse[0]:.2f}  RMSE(y)={rmse[1]:.2f}"
    )


report("train", Y_train, Y_hat_train)
report("val", Y_val, Y_hat_val)
print("(paper reports r ≈ 0.99 for Push-T positional probing on encoder latents)")

In [ ]:
# Quick visual: predicted vs true block xy on val
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, i, name in zip(axes, (0, 1), ("block_x", "block_y")):
    ax.scatter(Y_val[:, i], Y_hat_val[:, i], s=6, alpha=0.4)
    lims = [
        min(Y_val[:, i].min(), Y_hat_val[:, i].min()),
        max(Y_val[:, i].max(), Y_hat_val[:, i].max()),
    ]
    ax.plot(lims, lims, "k--", lw=1)
    ax.set_xlabel(f"true {name}")
    ax.set_ylabel(f"pred {name}")
    ax.set_title(name)
    ax.set_aspect("equal", adjustable="box")
fig.tight_layout()
fig.suptitle("Val: ridge probe on encoder latents", y=1.02)
plt.show()